In [ ]:
import os
print(os.listdir("./stock_market_dataset/Stocks")[:10])
print(os.listdir("./stock_market_dataset/ETFs")[:10])

In [ ]:
import pandas as pd
pddf = pd.read_csv("./stock_market_dataset/ETFs/xpp.us.txt")

In [ ]:
pddf.head()

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import pandas as pd
import os
from pathlib import Path


stocks_path = "./stock_market_dataset/Stocks"
etfs_path = "./stock_market_dataset/ETFs"

stock_files = [stocks_path+"/"+path for path in os.listdir(stocks_path)]
etfs_files = [etfs_path+"/"+path for path in os.listdir(etfs_path)]

def read_stocks(file_path):
    try:
        csv =  pd.read_csv(file_path)
        csv["company"] = Path(file_path).name
        csv["type"] = "stock"
        return csv
    except Exception as e:
        print(e)
        return None

stock_files_dfs = []
etfs_files_dfs = []

with ThreadPoolExecutor(max_workers=100) as executor:
    stock_files_dfs = list(executor.map(read_stocks,stock_files))

stock_files_dfs = [file for file in stock_files_dfs if file is not None]

len(stock_files_dfs)

def read_etfs(file_path):
    try:
        csv =  pd.read_csv(file_path)
        csv["company"] = Path(file_path).name
        csv["type"] = "etf"
        return csv
    except Exception as e:
        print(e)
        return None

with ThreadPoolExecutor(max_workers=100) as executor:
    etfs_files_dfs = list(executor.map(read_etfs,etfs_files))

etfs_files_dfs = [file for file in etfs_files_dfs if file is not None]

len(etfs_files_dfs)

In [ ]:
stock_files_dfs[100].head()

In [ ]:
stocks_df = pd.concat(stock_files_dfs,ignore_index=True) 

In [ ]:
stocks_df.iloc[14880000]

In [ ]:
close_stocks_df = stocks_df[["Date","Close","company"]].copy()

In [ ]:
close_stocks_min = close_stocks_df["Close"].min()
close_stocks_max = close_stocks_df["Close"].max()
print(close_stocks_min, close_stocks_max)

In [ ]:
close_stocks_df["Close_scaled"] = (close_stocks_df["Close"]-close_stocks_min)/(close_stocks_max-close_stocks_min)

In [ ]:
close_stocks_df.tail()

In [ ]:
close_stocks_df["DateType_Date"] = pd.to_datetime(close_stocks_df["Date"])

In [ ]:
stocks_start_date = close_stocks_df["DateType_Date"].min()
stocks_end_date = close_stocks_df["DateType_Date"].max()

In [ ]:
stocks_x_axis = (stocks_end_date-stocks_start_date).days
print(stocks_x_axis)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider

window_size = 500

fig, ax = plt.subplots(figsize=(12,6))
fig.subplots_adjust(bottom=0.25)



# Plot the initial 500 days
(line,) = ax.plot(stocks_x_axis, stocks_x_axis, color='#1f77b4', lw=1.5, label='Stock Price')

# Set strict Y-limits since your data is already normalized between 0 and 1
ax.set_ylim(-0.05, 1.05)
# Set the initial X-axis view (0 to 500)
ax.set_xlim(0, window_size)

# Add titles and grid labels
ax.set_title(
    'Single Company Stock Price - 20,000 Day Timeline',
    fontsize=14,
    pad=15,
    weight='bold',
)
ax.set_xlabel('Timeline (Days)', fontsize=11)
ax.set_ylabel('Normalized Price (0.0 - 1.0)', fontsize=11)
ax.grid(True, linestyle='--', alpha=0.5)

# 4. Create the slider axis [left, bottom, width, height] and the Slider widget
ax_slider = plt.axes([0.2, 0.08, 0.65, 0.04])
slider = Slider(
    ax_slider,
    label='Scroll Timeline ',
    valmin=0,
    valmax=stocks_x_axis - window_size,
    valinit=0,
    valstep=50,  # Moves by 50 days per click on the slider bar
    valfmt='%0.0f',
    color='#2ca02c',
)


# 5. Define what happens when the slider moves
def update_window(val):
  # Get the current slider position
  start_day = int(slider.val)

  # Shift the X-axis view window dynamically
  ax.set_xlim(start_day, start_day + window_size)

  # Redraw the canvas cleanly
  fig.canvas.draw_idle()


# Connect the slider to the update function
slider.on_changed(update_window)

plt.show()

In [ ]:
!pip install seaborn

In [ ]:
# ==========================================
# STEP 1: PIVOT STOCK DATASET (Date x Company)
# ==========================================

import pandas as pd
import numpy as np

print("Pivoting stocks dataframe...")

# Pivot so rows are unique dates and columns are stock company tickers
pivoted_stocks = close_stocks_df.pivot(
    index="DateType_Date", columns="company", values="Close"
).sort_index()

# Display matrix dimensions and full date range
print(f"Pivoted Shape: {pivoted_stocks.shape} (Total Dates x Total Companies)")
print(
    f"Timeline Range: {pivoted_stocks.index.min().strftime('%Y-%m-%d')} to"
    f" {pivoted_stocks.index.max().strftime('%Y-%m-%d')}"
)

# Rank companies by historical date completeness (number of active trading days)
completeness = pivoted_stocks.notna().sum().sort_values(ascending=False)
top_companies = completeness.head(25).index.tolist()

print("\nTop 10 Companies with longest historical record:")
for c in top_companies[:10]:
  print(" -", c.replace(".us.txt", "").upper())

# Preview the pivoted DataFrame
pivoted_stocks.head()


In [ ]:
# ==========================================
# STEP 2: BASE-100 PRICE NORMALIZATION
# ==========================================

# Formula: (Price at time t / Price on first trading day) * 100
# Every company starts at 100.0 on its first valid trading date
pivoted_norm = pivoted_stocks.apply(
    lambda col: (
        (col / col.dropna().iloc[0]) * 100 if not col.dropna().empty else col
    )
)

print("Base-100 Normalized Data Preview:")
pivoted_norm.head()


In [ ]:
# ==========================================
# STEP 3: PLOT ALL STOCKS (FULL TIMELINE)
# ==========================================

import matplotlib.pyplot as plt
import seaborn as sns

# Set clean aesthetic style
sns.set_theme(style="darkgrid")
fig, ax = plt.subplots(figsize=(15, 7))

# Sample 300 companies for background rendering to keep plot clean and fast
sample_cols = pivoted_norm.columns.to_series().sample(
    n=min(300, len(pivoted_norm.columns)), random_state=42
)

# Plot thin semi-transparent lines for sampled background stocks
for comp in sample_cols:
  ax.plot(
      pivoted_norm.index,
      pivoted_norm[comp],
      color="#7f8c8d",
      alpha=0.08,
      linewidth=0.8,
  )

# Calculate and plot the Overall Market Median Trend across all stocks
market_median = pivoted_norm.median(axis=1)
ax.plot(
    pivoted_norm.index,
    market_median,
    color="#e74c3c",
    linewidth=2.5,
    linestyle="--",
    label="Market Median Trend",
)

# Highlight top 8 companies with the longest trading history
palette = sns.color_palette("tab10", 8)
for idx, comp in enumerate(top_companies[:8]):
  ax.plot(
      pivoted_norm.index,
      pivoted_norm[comp],
      label=comp.replace(".us.txt", "").upper(),
      linewidth=2.0,
      color=palette[idx],
  )

# Logarithmic Y-axis scale to visualize multi-decade exponential growth
ax.set_yscale("log")

# Title and labels
ax.set_title(
    "All Stock Price Growth Trajectories Across Companies (Log Scale, Base ="
    " 100)",
    fontsize=15,
    pad=15,
    weight="bold",
)
ax.set_xlabel("Timeline (Date)", fontsize=12, weight="bold")
ax.set_ylabel(
    "Normalized Price Index (Log Scale)", fontsize=12, weight="bold"
)
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=True)

plt.tight_layout()
plt.show()


In [ ]:
# ==========================================
# STEP 4: INTERACTIVE TIMELINE SLIDER
# ==========================================

from matplotlib.widgets import Slider

# Select top 10 companies for clear window inspection
top_10 = top_companies[:10]
dates_array = pivoted_stocks.index
window_size_days = 500  # Number of trading days in window view

fig, ax = plt.subplots(figsize=(14, 6))
plt.subplots_adjust(bottom=0.25)

# Plot stock price lines
colors = plt.cm.tab10(np.linspace(0, 1, len(top_10)))
for idx, comp in enumerate(top_10):
  ax.plot(
      dates_array,
      pivoted_stocks[comp],
      label=comp.replace(".us.txt", "").upper(),
      lw=1.8,
      color=colors[idx],
  )

# Set initial view window (Day 0 to window_size_days)
ax.set_xlim(
    dates_array[0], dates_array[min(window_size_days, len(dates_array) - 1)]
)
ax.set_title(
    "Interactive Stock Price Explorer - Scrollable Timeline Window",
    fontsize=14,
    weight="bold",
    pad=15,
)
ax.set_xlabel("Date", fontsize=11, weight="bold")
ax.set_ylabel("Stock Close Price ($)", fontsize=11, weight="bold")
ax.grid(True, linestyle="--", alpha=0.5)
ax.legend(loc="upper left")

# Create slider axis widget below main plot
ax_slider = plt.axes([0.18, 0.08, 0.65, 0.04])
slider = Slider(
    ax_slider,
    label="Scroll Timeline ",
    valmin=0,
    valmax=len(dates_array) - window_size_days,
    valinit=0,
    valstep=30,  # Shifts window by 30 days per step
    color="#2ca02c",
)


# Define update function for slider movement
def update_timeline(val):
  start_idx = int(slider.val)
  end_idx = min(start_idx + window_size_days, len(dates_array) - 1)

  # Update X-axis range
  ax.set_xlim(dates_array[start_idx], dates_array[end_idx])

  # Dynamically adjust Y-axis scaling to fit highest and lowest price in current window
  visible_data = pivoted_stocks[top_10].iloc[start_idx:end_idx]
  if not visible_data.dropna(how="all").empty:
    ymax = visible_data.max().max()
    ymin = visible_data.min().min()
    if not np.isnan(ymin) and not np.isnan(ymax):
      ax.set_ylim(max(0, ymin * 0.9), ymax * 1.1)

  fig.canvas.draw_idle()


# Connect slider widget to update function
slider.on_changed(update_timeline)
plt.show()


In [ ]:
# ==========================================
# STEP 5: DAILY RETURNS CORRELATION HEATMAP
# ==========================================

# 1. Calculate daily percentage return for each stock
daily_returns = pivoted_stocks[top_companies[:25]].pct_change()

# 2. Compute Pearson correlation matrix
corr_matrix = daily_returns.corr()

# 3. Plot Correlation Heatmap
plt.figure(figsize=(14, 10))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",  # Red = Positive Correlation, Blue = Negative Correlation
    center=0,
    linewidths=0.5,
    cbar_kws={"label": "Correlation Coefficient"},
)

plt.title(
    "Stock Daily Return Correlation Matrix (Top 25 Traded Companies)",
    fontsize=15,
    weight="bold",
    pad=15,
)
plt.xticks(
    ticks=np.arange(len(top_companies[:25])) + 0.5,
    labels=[c.replace(".us.txt", "").upper() for c in top_companies[:25]],
    rotation=45,
    ha="right",
)
plt.yticks(
    ticks=np.arange(len(top_companies[:25])) + 0.5,
    labels=[c.replace(".us.txt", "").upper() for c in top_companies[:25]],
    rotation=0,
)
plt.tight_layout()
plt.show()


In [ ]:
# ==========================================
# STEP 6: CORRELATION DISTRIBUTION & TOP PAIRS (FIXED)
# ==========================================

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# 1. Calculate correlation matrix for top 100 companies
sample_100 = completeness.head(100).index
corr_100 = pivoted_stocks[sample_100].pct_change().corr()

# FIX: Rename index and columns to avoid duplicate 'company' level names when resetting index
corr_100.index.name = "Company_A"
corr_100.columns.name = "Company_B"

# 2. Extract upper triangle of correlation matrix (excluding self-correlations)
upper_mask = np.triu(np.ones(corr_100.shape), k=1).astype(bool)

# 3. Stack into a 2-column DataFrame of pairwise correlations
corr_pairs = corr_100.where(upper_mask).stack().reset_index()
corr_pairs.columns = ["Company_A", "Company_B", "Correlation"]

# Clean up ticker file extensions
corr_pairs["Company_A"] = (
    corr_pairs["Company_A"].str.replace(".us.txt", "").str.upper()
)
corr_pairs["Company_B"] = (
    corr_pairs["Company_B"].str.replace(".us.txt", "").str.upper()
)

# 4. Plot Market-Wide Correlation Distribution
fig, ax = plt.subplots(figsize=(11, 5))
sns.histplot(
    corr_pairs["Correlation"],
    kde=True,
    color="#2980b9",
    bins=50,
    ax=ax,
    stat="density",
)

mean_corr = corr_pairs["Correlation"].mean()
ax.axvline(
    mean_corr,
    color="red",
    linestyle="--",
    linewidth=2,
    label=f"Mean Market Correlation: {mean_corr:.3f}",
)

ax.set_title(
    "Market-Wide Stock Daily Return Correlation Distribution",
    fontsize=14,
    weight="bold",
    pad=15,
)
ax.set_xlabel("Pearson Correlation Coefficient (r)", fontsize=11, weight="bold")
ax.set_ylabel("Density", fontsize=11, weight="bold")
ax.legend()
plt.tight_layout()
plt.show()

# 5. Display Top 10 Positively Correlated Pairs
print("=== Top 10 Most Positively Correlated Stock Pairs ===")
display(corr_pairs.sort_values(by="Correlation", ascending=False).head(10))

# 6. Display Top 10 Negatively / Least Correlated Pairs
print("\n=== Top 10 Least / Most Negatively Correlated Stock Pairs ===")
display(corr_pairs.sort_values(by="Correlation", ascending=True).head(10))


In [ ]:
# ==========================================
# STEP 7: FAST TIMELINE GRAPH NAVIGATOR
# Uses pre-computed 'pivoted_stocks' & 'top_companies' from memory
# ==========================================

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# ---------------------------------------------------------
# CHANGE THESE 2 VARIABLES TO NAVIGATE THE TIMELINE INSTANTLY
# ---------------------------------------------------------
start_day = 8000  # Day offset from timeline start (e.g., 0, 500, 1500, 3000...)
window_size = 10000  # Number of trading days to display in the window

# Select top 10 companies from memory
top_10 = top_companies[:10]
dates_array = pivoted_stocks.index
total_days = len(dates_array)

# Calculate slice bounds safely
start_idx = min(max(0, start_day), total_days - 50)
end_idx = min(start_idx + window_size, total_days - 1)

# FAST SLICE: Slice existing DataFrame in memory
visible_dates = dates_array[start_idx:end_idx]
window_data = pivoted_stocks[top_10].iloc[start_idx:end_idx]

# ---------------------------------------------------------
# FAST PLOT RENDERING
# ---------------------------------------------------------
sns.set_theme(style="darkgrid")
fig, ax = plt.subplots(figsize=(14, 6))

colors = plt.cm.tab10(np.linspace(0, 1, len(top_10)))
for idx, comp in enumerate(top_10):
  ax.plot(
      visible_dates,
      window_data[comp],
      label=comp.replace(".us.txt", "").upper(),
      lw=1.8,
      color=colors[idx],
  )

start_date_str = visible_dates[0].strftime("%Y-%m-%d")
end_date_str = visible_dates[-1].strftime("%Y-%m-%d")

ax.set_title(
    f"Stock Price Timeline (Days {start_idx} to {end_idx}: {start_date_str} to"
    f" {end_date_str})",
    fontsize=14,
    weight="bold",
    pad=15,
)
ax.set_xlabel("Date", fontsize=11, weight="bold")
ax.set_ylabel("Stock Close Price ($)", fontsize=11, weight="bold")
ax.legend(loc="upper left")

plt.tight_layout()
plt.show()


In [ ]:
# ==========================================
# STEP 8: VARIABLE-BASED COMPANY RANGE & TIMELINE NAVIGATOR
# Uses pre-computed 'pivoted_stocks' & 'top_companies' from memory
# ==========================================

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# ---------------------------------------------------------
# 1. SET YOUR TIMELINE RANGE VARIABLES
# ---------------------------------------------------------
start_day = 0  # Offset from timeline start in trading days (e.g. 0, 500, 2000...)
window_size = 20000  # Number of trading days to display in window (e.g. 200, 500, 1000)

# ---------------------------------------------------------
# 2. SET YOUR COMPANY RANGE / SELECTION VARIABLES
# ---------------------------------------------------------
# Set USE_CUSTOM_TICKERS = False to select a range of companies by rank/index
# Set USE_CUSTOM_TICKERS = True to specify exact ticker names
USE_CUSTOM_TICKERS = False

# Option A: Company Range Settings (when USE_CUSTOM_TICKERS = False)
company_start_idx = 0  # Starting rank of company (0 = 1st company, 10 = 11th company...)
num_companies = 50  # How many companies to display from company_start_idx

# Option B: Specific Ticker Names (when USE_CUSTOM_TICKERS = True)
custom_tickers = ["aapl", "msft", "ibm", "ge", "xom", "jnj", "wmt", "cvx"]

# ---------------------------------------------------------
# 3. FAST DATA SLICING (IN MEMORY)
# ---------------------------------------------------------
# Filter Companies
if USE_CUSTOM_TICKERS:
  available_cols = pivoted_stocks.columns
  selected_cols = []
  for ticker in custom_tickers:
    matches = [
        c
        for c in available_cols
        if ticker.lower() in c.lower().replace(".us.txt", "")
    ]
    if matches:
      selected_cols.extend(matches)
  selected_cols = list(dict.fromkeys(selected_cols))  # Remove duplicates
else:
  # Slice company columns by range from ranked completeness
  selected_cols = top_companies[
      company_start_idx : company_start_idx + num_companies
  ]

# Filter Timeline Dates
dates_array = pivoted_stocks.index
total_days = len(dates_array)
start_idx = min(max(0, start_day), total_days - 50)
end_idx = min(start_idx + window_size, total_days - 1)

# Slice DataFrame in memory (< 0.01 seconds)
visible_dates = dates_array[start_idx:end_idx]
window_data = pivoted_stocks[selected_cols].iloc[start_idx:end_idx]

# ---------------------------------------------------------
# 4. PLOT RENDERING
# ---------------------------------------------------------
sns.set_theme(style="darkgrid")
fig, ax = plt.subplots(figsize=(15, 6))

# Generate distinct colors for selected companies
colors = plt.cm.tab10(np.linspace(0, 1, max(1, len(selected_cols))))
for idx, comp in enumerate(selected_cols):
  ax.plot(
      visible_dates,
      window_data[comp],
      label=comp.replace(".us.txt", "").upper(),
      lw=1.8,
      color=colors[idx % 10],
  )

start_date_str = visible_dates[0].strftime("%Y-%m-%d")
end_date_str = visible_dates[-1].strftime("%Y-%m-%d")

# Add title and labels
ax.set_title(
    f"Stock Prices: {start_date_str} to {end_date_str} | Companies"
    f" {company_start_idx + 1} to {company_start_idx + len(selected_cols)}",
    fontsize=14,
    weight="bold",
    pad=15,
)
ax.set_xlabel("Date", fontsize=11, weight="bold")
ax.set_ylabel("Stock Close Price ($)", fontsize=11, weight="bold")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=True)

plt.tight_layout()
plt.show()


In [ ]:
# ==========================================
# STEP 9: VARIABLE-BASED COMPANY RANGE & TIMELINE NAVIGATOR
# Uses pre-computed 'pivoted_stocks' & 'completeness' from memory
# ==========================================

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

completeness_indexes = completeness.index.to_list()

# ---------------------------------------------------------
# 1. SET YOUR TIMELINE RANGE VARIABLES
# ---------------------------------------------------------
start_day = 0  # Offset from timeline start in trading days (e.g. 0, 500, 2000...)
window_size = 20000  # Number of trading days to display in window (e.g. 200, 500, 1000)

# ---------------------------------------------------------
# 2. SET YOUR COMPANY RANGE / SELECTION VARIABLES
# ---------------------------------------------------------
# Set USE_CUSTOM_TICKERS = False to select a range of companies by rank/index
# Set USE_CUSTOM_TICKERS = True to specify exact ticker names
USE_CUSTOM_TICKERS = False

# Option A: Company Range Settings (when USE_CUSTOM_TICKERS = False)
company_start_idx = 15  # Starting rank of company (0 = 1st company, 10 = 11th company...)
num_companies = 10  # How many companies to display from company_start_idx

# Option B: Specific Ticker Names (when USE_CUSTOM_TICKERS = True)
custom_tickers = ["aapl", "msft", "ibm", "ge", "xom", "jnj", "wmt", "cvx"]

# ---------------------------------------------------------
# 3. FAST DATA SLICING (IN MEMORY)
# ---------------------------------------------------------
# Filter Companies
if USE_CUSTOM_TICKERS:
  available_cols = pivoted_stocks.columns
  selected_cols = []
  for ticker in custom_tickers:
    matches = [
        c
        for c in available_cols
        if ticker.lower() in c.lower().replace(".us.txt", "")
    ]
    if matches:
      selected_cols.extend(matches)
  selected_cols = list(dict.fromkeys(selected_cols))  # Remove duplicates
else:
  # Slice company columns by range from ranked completeness
  selected_cols = completeness_indexes[
      company_start_idx : company_start_idx + num_companies
  ]

# Filter Timeline Dates
dates_array = pivoted_stocks.index
total_days = len(dates_array)
start_idx = min(max(0, start_day), total_days - 50)
end_idx = min(start_idx + window_size, total_days - 1)

# Slice DataFrame in memory (< 0.01 seconds)
visible_dates = dates_array[start_idx:end_idx]
window_data = pivoted_stocks[selected_cols].iloc[start_idx:end_idx]

# ---------------------------------------------------------
# 4. PLOT RENDERING
# ---------------------------------------------------------
sns.set_theme(style="darkgrid")
fig, ax = plt.subplots(figsize=(15, 6))

# Generate distinct colors for selected companies
colors = plt.cm.tab10(np.linspace(0, 1, max(1, len(selected_cols))))
for idx, comp in enumerate(selected_cols):
  ax.plot(
      visible_dates,
      window_data[comp],
      label=comp.replace(".us.txt", "").upper(),
      lw=1.8,
      color=colors[idx % 10],
  )

start_date_str = visible_dates[0].strftime("%Y-%m-%d")
end_date_str = visible_dates[-1].strftime("%Y-%m-%d")

# Add title and labels
ax.set_title(
    f"Stock Prices: {start_date_str} to {end_date_str} | Companies"
    f" {company_start_idx + 1} to {company_start_idx + len(selected_cols)}",
    fontsize=14,
    weight="bold",
    pad=15,
)
ax.set_xlabel("Date", fontsize=11, weight="bold")
ax.set_ylabel("Stock Close Price ($)", fontsize=11, weight="bold")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=True)

plt.tight_layout()
plt.show()


## Event Analysis through stocks ##

In [ ]:
!pip install pandas-datareader

In [ ]:
# ==============================================================================
# STEP 1: HISTORICAL MARKET EVENTS, WARS, AND BUBBLE DEFINITIONS
# ==============================================================================
import pandas as pd
import numpy as np

# Dictionary of major macroeconomic events, wars, and financial bubble periods
MARKET_EVENTS = {
    "1973 Oil Crisis": {
        "start": "1973-10-01", "end": "1974-03-31", 
        "category": "Oil Shock", "color": "#ff9999", "alpha": 0.35
    },
    "1987 Black Monday Crash": {
        "start": "1987-10-01", "end": "1987-12-31", 
        "category": "Market Crash", "color": "#e6b800", "alpha": 0.4
    },
    "1990 Gulf War / Recession": {
        "start": "1990-07-01", "end": "1991-03-31", 
        "category": "War / Recession", "color": "#c2c2f0", "alpha": 0.4
    },
    "1997 Asian Financial Crisis": {
        "start": "1997-07-01", "end": "1998-01-31", 
        "category": "Financial Crisis", "color": "#ffcc99", "alpha": 0.4
    },
    "Dot-Com Bubble Peak & Pop": {
        "start": "1999-01-01", "end": "2002-10-01", 
        "category": "Bubble / Tech Crash", "color": "#ff6666", "alpha": 0.3
    },
    "9/11 Shock & Iraq War": {
        "start": "2001-09-01", "end": "2003-05-01", 
        "category": "War / Geopolitical", "color": "#b3b3cc", "alpha": 0.4
    },
    "2008 Real Estate Crisis": {
        "start": "2007-10-01", "end": "2009-03-01", 
        "category": "Bubble / Credit Collapse", "color": "#ff3333", "alpha": 0.35
    },
    "2011 US Debt Ceiling Shock": {
        "start": "2011-07-01", "end": "2011-10-01", 
        "category": "Sovereign Crisis", "color": "#d9b3ff", "alpha": 0.4
    },
    "2015-16 Oil & China Crash": {
        "start": "2015-06-01", "end": "2016-02-01", 
        "category": "Commodity Shock", "color": "#ffc266", "alpha": 0.4
    }
}

# Convert event dates to pandas Datetime objects
events_df = pd.DataFrame.from_dict(MARKET_EVENTS, orient='index')
events_df['start'] = pd.to_datetime(events_df['start'])
events_df['end'] = pd.to_datetime(events_df['end'])

print("=== Market Events & Bubbles Registry ===")
display(events_df[['category', 'start', 'end']])


In [ ]:
# ==============================================================================
# STEP 2: CPI INFLATION ADJUSTMENT ENGINE
# ==============================================================================
import pandas_datareader.data as web

def fetch_or_build_cpi_index(dates_index):
    """
    Fetches monthly US CPI (CPIAUCSL) from FRED or constructs an accurate historical 
    CPI baseline matching the dataset's exact date range (1962 to present).
    """
    start_date = dates_index.min()
    end_date = dates_index.max()
    
    try:
        # Attempt to pull authentic US CPI from Federal Reserve Economic Data (FRED)
        cpi = web.DataReader('CPIAUCSL', 'fred', start_date, end_date)
        cpi = cpi.rename(columns={'CPIAUCSL': 'CPI'})
    except Exception as e:
        print(f"Notice: FRED API unavailable ({e}). Using historical monthly benchmark CPI curve.")
        # Historical US CPI baseline values for key anchor years
        cpi_anchors = {
            '1962-01-01': 30.0,  '1970-01-01': 37.8,  '1980-01-01': 77.8,
            '1990-01-01': 127.4, '2000-01-01': 168.8, '2008-01-01': 211.08,
            '2017-11-01': 246.66
        }
        anchor_df = pd.DataFrame(list(cpi_anchors.items()), columns=['Date', 'CPI'])
        anchor_df['Date'] = pd.to_datetime(anchor_df['Date'])
        anchor_df = anchor_df.set_index('Date')
        
        # Resample monthly and interpolate smoothly
        cpi = anchor_df.resample('MS').interpolate(method='linear')

    # Reindex CPI to match daily stock trading calendar dates via linear interpolation
    daily_cpi = cpi.reindex(dates_index).interpolate(method='linear').bfill().ffill()
    
    # Normalize CPI relative to the latest date (Base = Latest purchasing power)
    latest_cpi = daily_cpi['CPI'].iloc[-1]
    daily_cpi['Inflation_Multiplier'] = latest_cpi / daily_cpi['CPI']
    
    return daily_cpi

# Example using your pivoted close price dataframe index (DateType_Date)
# Assuming `pivoted_stocks` or `close_stocks_df` contains daily stock data with Datetime index:
if 'DateType_Date' in close_stocks_df.columns:
    unique_dates = pd.DatetimeIndex(close_stocks_df['DateType_Date'].unique()).sort_values()
    cpi_df = fetch_or_build_cpi_index(unique_dates)
    
    # Merge CPI multiplier into close prices
    close_stocks_df = close_stocks_df.merge(
        cpi_df[['Inflation_Multiplier']], 
        left_on='DateType_Date', 
        right_index=True, 
        how='left'
    )
    
    # Calculate Inflation-Adjusted (Real 2017-Dollar) Price
    close_stocks_df['Real_Close'] = close_stocks_df['Close'] * close_stocks_df['Inflation_Multiplier']
    print("Inflation adjustment complete!")
    display(close_stocks_df[['Date', 'company', 'Close', 'Inflation_Multiplier', 'Real_Close']].head())


In [ ]:
# ==============================================================================
# STEP 3: INFLATION-NEUTRAL CORRELATION MATRIX GENERATOR
# ==============================================================================
import seaborn as sns
import matplotlib.pyplot as plt

def compute_inflation_neutral_correlation(pivoted_prices_df, sample_tickers=None):
    """
    Computes and plots:
    1. Raw Price Correlation (distorted by inflation trend)
    2. Inflation-Neutral Log Return Correlation (pure co-movement)
    """
    if sample_tickers:
        prices = pivoted_prices_df[sample_tickers].dropna(how='all')
    else:
        prices = pivoted_prices_df.dropna(axis=1, thresh=len(pivoted_prices_df)*0.7)

    # 1. Raw Price Correlation Matrix (Biased by Long-Term Inflation)
    raw_corr = prices.corr()
    
    # 2. Log Returns (Detrended & Inflation-Neutral)
    log_returns = np.log(prices / prices.shift(1))
    neutral_corr = log_returns.corr()
    
    # Visualization: Comparing Raw vs Inflation-Neutral Correlation
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    
    sns.heatmap(raw_corr, ax=axes[0], cmap='vlag', vmin=-1, vmax=1, annot=True, fmt=".2f")
    axes[0].set_title("Raw Price Correlation\n(Inflated by Long-Term Trend Bias)", fontsize=12, fontweight='bold')
    
    sns.heatmap(neutral_corr, ax=axes[1], cmap='vlag', vmin=-1, vmax=1, annot=True, fmt=".2f")
    axes[1].set_title("Inflation-Neutral Log-Return Correlation\n(Pure Co-Movement)", fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    return raw_corr, neutral_corr

# Usage Example: Choose a sample list of classic stocks/ETFs from your dataset
sample_list = ['ibm.us.txt', 'ge.us.txt', 'mcd.us.txt', 'cat.us.txt', 'pg.us.txt']
available_samples = [c for c in sample_list if c in close_stocks_df['company'].unique()]

# Pivot table for sample
pivoted_sample = close_stocks_df[close_stocks_df['company'].isin(available_samples)].pivot(
    index='DateType_Date', columns='company', values='Close'
)

raw_corr, neutral_corr = compute_inflation_neutral_correlation(pivoted_sample)


In [ ]:
# ==============================================================================
# STEP 4: TIMELINE VISUALIZATION WITH BUBBLE & WAR OVERLAYS (FIXED)
# ==============================================================================
import pandas as pd
import matplotlib.pyplot as plt

def plot_stock_with_events_and_inflation(stock_df, ticker_name):
    """
    Plots Nominal Price vs Real (Inflation-Adjusted) Price with shaded historical bubbles/events.
    """
    # 1. Filter and prepare data
    df = stock_df[stock_df['company'] == ticker_name].copy()
    
    if df.empty:
        print(f"Warning: Ticker '{ticker_name}' not found in dataset.")
        return

    # 2. Ensure date column is datetime format before setting index
    if 'DateType_Date' in df.columns:
        df['DateType_Date'] = pd.to_datetime(df['DateType_Date'])
        df.set_index('DateType_Date', inplace=True)
    elif 'Date' in df.columns:
        df['Date'] = pd.to_datetime(df['Date'])
        df.set_index('Date', inplace=True)
        
    df.sort_index(inplace=True)
    
    fig, ax = plt.subplots(figsize=(15, 7))
    
    # 3. Plot Nominal and Real Prices
    ax.plot(df.index, df['Close'], label='Nominal Close Price ($)', color='#1f77b4', linewidth=1.5)
    if 'Real_Close' in df.columns:
        ax.plot(df.index, df['Real_Close'], label='Real Close Price (Inflation-Adjusted)', 
                color='#2ca02c', linestyle='--', linewidth=1.5)
    
    # Get dataset bounds as Timestamps
    min_date = df.index.min()
    max_date = df.index.max()
    
    # 4. Overlay Event Shading with Explicit pd.to_datetime Conversion
    for event_name, info in MARKET_EVENTS.items():
        event_start = pd.to_datetime(info['start'])
        event_end = pd.to_datetime(info['end'])
        
        # Check date overlap using Timestamps
        if event_start <= max_date and event_end >= min_date:
            ax.axvspan(event_start, event_end, color=info['color'], alpha=info['alpha'], label=event_name)
            
            # Position label cleanly at the midpoint of event range within plot bounds
            plot_event_start = max(event_start, min_date)
            plot_event_end = min(event_end, max_date)
            mid_date = plot_event_start + (plot_event_end - plot_event_start) / 2
            
            ax.text(mid_date, ax.get_ylim()[1] * 0.90, event_name, rotation=45, 
                    fontsize=8, fontweight='bold', ha='center', va='top', 
                    bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="gray", alpha=0.7))

    ax.set_title(f'Historical Trajectory & Event Impact: {ticker_name.upper()}', fontsize=14, fontweight='bold', pad=15)
    ax.set_xlabel('Year', fontsize=12)
    ax.set_ylabel('Price ($)', fontsize=12)
    ax.grid(True, linestyle=':', alpha=0.6)
    
    # Consolidate duplicate legend entries
    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ax.legend(by_label.values(), by_label.keys(), loc='upper left', fontsize=9, framealpha=0.9)
    
    plt.tight_layout()
    plt.show()

# Run visualization on IBM
plot_stock_with_events_and_inflation(close_stocks_df, 'msft.us.txt')


In [ ]:
# import os
# stock_files = os.listdir("./stock_market_dataset/Stocks")
# for file in stock_files:
#     plot_stock_with_events_and_inflation(close_stocks_df, str(file))


In [ ]:
# ==============================================================================
# STEP 5: REGIME-BASED CORRELATION ANALYSIS (CRISIS VS NORMAL)
# ==============================================================================
def analyze_crisis_correlations(pivoted_prices_df):
    """
    Calculates cross-asset return correlations during specific historical regimes:
    - 2000 Dot-Com Bubble Pop
    - 2008 Real Estate Crisis
    - Normal Growth Periods
    """
    log_returns = np.log(pivoted_prices_df / pivoted_prices_df.shift(1))
    
    regimes = {
        "Dot-Com Crash (1999-2002)": ('1999-01-01', '2002-10-01'),
        "2008 Real Estate Crisis": ('2007-10-01', '2009-03-01'),
        "Post-Crisis Recovery (2012-2016)": ('2012-01-01', '2016-12-31')
    }
    
    fig, axes = plt.subplots(1, len(regimes), figsize=(18, 5))
    
    for idx, (name, (start, end)) in enumerate(regimes.items()):
        subset_returns = log_returns.loc[start:end].dropna(axis=1, how='all')
        corr = subset_returns.corr()
        
        # Calculate average pairwise correlation (excluding self-correlation = 1)
        mask = np.ones(corr.shape, dtype=bool)
        np.fill_diagonal(mask, 0)
        avg_corr = corr.values[mask].mean()
        
        sns.heatmap(corr, ax=axes[idx], cmap='coolwarm', vmin=-0.2, vmax=1.0, cbar=True)
        axes[idx].set_title(f"{name}\nAvg Pairwise Corr: {avg_corr:.2f}", fontsize=11, fontweight='bold')

    plt.tight_layout()
    plt.show()

# Run correlation comparison across historical regimes
if not pivoted_sample.empty:
    analyze_crisis_correlations(pivoted_sample)


In [ ]:
!pip install scipy

In [ ]:
# ==============================================================================
# TIME-SLICED VECTORIZED CORRELATION ENGINE (5-YEAR WINDOW COMBINER)
# ==============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def compute_time_sliced_correlation(close_stocks_df, window_years=5):
    """
    Slices the multi-decade timeline into clean N-year windows, computes vectorized
    correlation matrices for each window in C/NumPy, and combines them into a master
    time-overlapping correlation matrix.
    """
    print(f"1. Preparing data and creating {window_years}-year time slices...")
    
    price_col = 'Real_Close' if 'Real_Close' in close_stocks_df.columns else 'Close'
    date_col = 'DateType_Date' if 'DateType_Date' in close_stocks_df.columns else 'Date'

    # Create master pivoted prices
    pivoted_prices = close_stocks_df.pivot(index=date_col, columns='company', values=price_col)
    pivoted_prices.index = pd.to_datetime(pivoted_prices.index)
    pivoted_prices.sort_index(inplace=True)
    
    # Calculate daily log returns
    log_returns = np.log(pivoted_prices / pivoted_prices.shift(1))
    
    # Filter stocks with at least 500 total active days across timeline
    valid_tickers = log_returns.count()[log_returns.count() >= 500].index
    returns_clean = log_returns[valid_tickers]
    
    # Generate timeframe slices (e.g. 1970-1975, 1975-1980, ..., 2015-2020)
    min_year = returns_clean.index.min().year
    max_year = returns_clean.index.max().year
    
    slice_corrs = []
    
    print(f"2. Processing time-slice windows from {min_year} to {max_year}...")
    for start_yr in range(min_year, max_year, window_years):
        end_yr = start_yr + window_years
        slice_mask = (returns_clean.index.year >= start_yr) & (returns_clean.index.year < end_yr)
        slice_df = returns_clean.loc[slice_mask]
        
        # Keep stocks that have at least 60% active data within this 5-year window
        min_active_days = int(0.60 * len(slice_df))
        active_in_slice = slice_df.count()[slice_df.count() >= min_active_days].index
        
        if len(active_in_slice) > 1:
            # FAST Vectorized Correlation on clean slice
            sub_returns = slice_df[active_in_slice].fillna(0).values
            # Standardize
            stds = sub_returns.std(axis=0)
            stds[stds == 0] = 1e-8
            norm_sub = (sub_returns - sub_returns.mean(axis=0)) / stds
            
            # Matrix dot product (BLAS C-level speed)
            corr_mat = np.dot(norm_sub.T, norm_sub) / norm_sub.shape[0]
            
            slice_corr_df = pd.DataFrame(corr_mat, index=active_in_slice, columns=active_in_slice)
            slice_corrs.append(slice_corr_df)
            
    print(f"3. Combining {len(slice_corrs)} time-slice correlation matrices...")
    
    # Reindex all slices to master ticker grid and take mean across valid overlapping time windows
    master_corr = pd.concat(slice_corrs).groupby(level=0).mean()
    master_corr = master_corr.reindex(columns=master_corr.index)
    
    print("Done! Master time-sliced correlation matrix generated.")
    return master_corr, pivoted_prices

# Run the time-sliced engine
master_corr_matrix, pivoted_prices = compute_time_sliced_correlation(close_stocks_df, window_years=5)


In [ ]:
# ==============================================================================
# SELF-CONTAINED CONFIGURABLE BUCKET LOOP (WITH AUTO-CLUSTERING)
# ==============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

# ------------------------------------------------------------------------------
# 1. ENSURE BUCKETING & PIVOTED DATA EXISTS
# ------------------------------------------------------------------------------
# Auto-generate master_corr_matrix if missing
if 'master_corr_matrix' not in globals() or 'pivoted_prices' not in globals():
    print("Building master time-sliced correlation matrix...")
    price_col = 'Real_Close' if 'Real_Close' in close_stocks_df.columns else 'Close'
    date_col = 'DateType_Date' if 'DateType_Date' in close_stocks_df.columns else 'Date'
    
    pivoted_prices = close_stocks_df.pivot(index=date_col, columns='company', values=price_col)
    pivoted_prices.index = pd.to_datetime(pivoted_prices.index)
    pivoted_prices.sort_index(inplace=True)
    
    # Fast daily returns for top 300 stocks
    valid_tickers = pivoted_prices.count()[pivoted_prices.count() >= 500].head(300).index
    pivoted_prices = pivoted_prices[valid_tickers]
    log_returns = np.log(pivoted_prices / pivoted_prices.shift(1))
    
    # Vectorized Correlation
    norm_ret = (log_returns.fillna(0) - log_returns.fillna(0).mean()) / (log_returns.fillna(0).std() + 1e-8)
    fast_corr = np.dot(norm_ret.T, norm_ret) / len(norm_ret)
    master_corr_matrix = pd.DataFrame(fast_corr, index=valid_tickers, columns=valid_tickers)

# Auto-generate stock_bucket_df if missing
if 'stock_bucket_df' not in globals():
    print("Generating stock co-movement buckets...")
    N_BUCKETS_TOTAL = 5
    clean_corr = master_corr_matrix.fillna(0)
    dist_matrix = np.sqrt(np.maximum(0, 2 * (1 - clean_corr.values)))
    condensed_dist = squareform(dist_matrix, checks=False)
    
    linkage_mat = linkage(condensed_dist, method='ward')
    cluster_labels = fcluster(linkage_mat, t=N_BUCKETS_TOTAL, criterion='maxclust')
    
    stock_bucket_df = pd.DataFrame({
        'company': master_corr_matrix.index,
        'Bucket': cluster_labels
    }).set_index('company')

# ------------------------------------------------------------------------------
# 2. PARAMETERIZABLE LOOP VISUALIZATION
# ------------------------------------------------------------------------------
def run_bucket_visualizer_loop(start_bucket=1, num_buckets=3, max_companies_per_plot=10):
    all_buckets = sorted(stock_bucket_df['Bucket'].unique())
    target_buckets = [b for b in all_buckets if b >= start_bucket][:num_buckets]
    
    print(f"=== Rendering {len(target_buckets)} Bucket Graph(s) starting at Bucket #{start_bucket} ===")
    
    for bucket_id in target_buckets:
        bucket_tickers = stock_bucket_df[stock_bucket_df['Bucket'] == bucket_id].index.tolist()
        selected_tickers = bucket_tickers[:max_companies_per_plot]
        
        # Determine longest-lived company for full plot bounds
        valid_series = {}
        earliest_date = pd.Timestamp.max
        latest_date = pd.Timestamp.min
        longest_lived_ticker = ""
        max_duration = -1
        
        for ticker in selected_tickers:
            if ticker in pivoted_prices.columns:
                s = pivoted_prices[ticker].dropna()
                if not s.empty:
                    valid_series[ticker] = s
                    duration = (s.index.max() - s.index.min()).days
                    if duration > max_duration:
                        max_duration = duration
                        longest_lived_ticker = ticker
                    earliest_date = min(earliest_date, s.index.min())
                    latest_date = max(latest_date, s.index.max())

        if not valid_series:
            continue

        fig, ax = plt.subplots(figsize=(16, 7))
        colors = cm.turbo(np.linspace(0.15, 0.85, len(valid_series)))
        
        for idx, (ticker, series) in enumerate(valid_series.items()):
            clean_name = ticker.replace('.us.txt', '').upper()
            ipo_year = series.index.min().year
            
            if ticker == longest_lived_ticker:
                label_txt = f"★ ANCHOR (Longest Lived): {clean_name} ({ipo_year}-{series.index.max().year})"
                lw, z_order = 2.8, 10
            else:
                label_txt = f"{clean_name} (Listed {ipo_year})"
                lw, z_order = 1.5, 5
                
            base_100 = (series / series.iloc[0]) * 100
            ax.plot(base_100.index, base_100, label=label_txt, color=colors[idx], linewidth=lw, zorder=z_order, alpha=0.85)

        # Overlay Market Events, Bubbles & Wars
        if 'MARKET_EVENTS' in globals():
            for event_name, info in MARKET_EVENTS.items():
                ev_start = pd.to_datetime(info['start'])
                ev_end = pd.to_datetime(info['end'])
                if ev_start <= latest_date and ev_end >= earliest_date:
                    ax.axvspan(ev_start, ev_end, color=info['color'], alpha=0.18)
                    mid_date = max(ev_start, earliest_date) + (min(ev_end, latest_date) - max(ev_start, earliest_date)) / 2
                    ax.text(
                        mid_date, ax.get_ylim()[1] * 0.85, event_name, rotation=90, 
                        fontsize=8, fontweight='bold', ha='center', va='top',
                        bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="gray", alpha=0.75)
                    )

        ax.set_xlim(earliest_date, latest_date)
        ax.set_yscale('log')
        longest_clean = longest_lived_ticker.replace('.us.txt', '').upper()
        ax.set_title(f'CO-MOVEMENT BUCKET #{bucket_id} | Anchor: {longest_clean} ({earliest_date.year}-{latest_date.year})', 
                     fontsize=14, fontweight='bold', pad=15)
        ax.set_xlabel('Timeline Year', fontsize=12)
        ax.set_ylabel('Re-indexed Price (Base 100, Log Scale)', fontsize=11)
        ax.grid(True, linestyle=':', alpha=0.6)
        ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1), fontsize=9, title="Correlated Group", framealpha=0.95)

        plt.tight_layout()
        plt.show()

# ==============================================================================
# CONTROLLABLE LOOP INPUTS (MODIFY THESE TWO VALUES TO NAVIGATE BUCKETS)
# ==============================================================================
START_BUCKET = 1      # Change to start from a different bucket (e.g. 1, 2, 3...)
NUM_ITERATIONS = 200    # Number of bucket graphs to show in this run

run_bucket_visualizer_loop(start_bucket=START_BUCKET, num_buckets=NUM_ITERATIONS, max_companies_per_plot=10)


In [ ]:
# ==============================================================================
# BUCKET & COMPANY PAGINATED VISUALIZER
# ==============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import math

def run_bucket_visualizer_paginated(
    start_bucket=1, 
    num_buckets=1, 
    company_page=1, 
    companies_per_page=10
):
    """
    Renders co-movement bucket graphs with full dual-pagination:
    - Bucket Pagination (start_bucket, num_buckets)
    - Company Pagination (company_page, companies_per_page)
    """
    all_buckets = sorted(stock_bucket_df['Bucket'].unique())
    target_buckets = [b for b in all_buckets if b >= start_bucket][:num_buckets]
    
    if not target_buckets:
        print(f"Warning: No buckets found starting at #{start_bucket}. (Available: {all_buckets})")
        return

    print(f"=== Rendering {len(target_buckets)} Bucket Graph(s) | Company Page {company_page} ({companies_per_page} per page) ===")
    
    for bucket_id in target_buckets:
        bucket_tickers = stock_bucket_df[stock_bucket_df['Bucket'] == bucket_id].index.tolist()
        total_companies_in_bucket = len(bucket_tickers)
        
        if total_companies_in_bucket == 0:
            continue
            
        # Calculate pagination slice for companies inside this bucket
        total_pages = math.ceil(total_companies_in_bucket / companies_per_page)
        
        # Clamp company_page to valid range
        current_page = max(1, min(company_page, total_pages))
        
        start_idx = (current_page - 1) * companies_per_page
        end_idx = min(start_idx + companies_per_page, total_companies_in_bucket)
        
        selected_tickers = bucket_tickers[start_idx:end_idx]
        
        # Determine longest-lived company across the ENTIRE bucket for fixed time bounds
        valid_series = {}
        earliest_date = pd.Timestamp.max
        latest_date = pd.Timestamp.min
        longest_lived_ticker = ""
        max_duration = -1
        
        # First scan all tickers in bucket to find timeline bounds
        for ticker in bucket_tickers:
            if ticker in pivoted_prices.columns:
                s = pivoted_prices[ticker].dropna()
                if not s.empty:
                    duration = (s.index.max() - s.index.min()).days
                    if duration > max_duration:
                        max_duration = duration
                        longest_lived_ticker = ticker
                    earliest_date = min(earliest_date, s.index.min())
                    latest_date = max(latest_date, s.index.max())

        # Collect series for the current page slice
        page_series = {}
        for ticker in selected_tickers:
            if ticker in pivoted_prices.columns:
                s = pivoted_prices[ticker].dropna()
                if not s.empty:
                    page_series[ticker] = s

        if not page_series:
            print(f"No active data for Bucket #{bucket_id} on Company Page {current_page}.")
            continue

        # Render Figure
        fig, ax = plt.subplots(figsize=(16, 7))
        colors = cm.turbo(np.linspace(0.15, 0.85, len(page_series)))
        
        for idx, (ticker, series) in enumerate(page_series.items()):
            clean_name = ticker.replace('.us.txt', '').upper()
            ipo_year = series.index.min().year
            
            if ticker == longest_lived_ticker:
                label_txt = f"★ ANCHOR (Longest Lived): {clean_name} ({ipo_year}-{series.index.max().year})"
                lw, z_order = 2.8, 10
            else:
                label_txt = f"{clean_name} (Listed {ipo_year})"
                lw, z_order = 1.5, 5
                
            base_100 = (series / series.iloc[0]) * 100
            ax.plot(base_100.index, base_100, label=label_txt, color=colors[idx], linewidth=lw, zorder=z_order, alpha=0.85)

        # Overlay Market Events, Bubbles & Wars
        if 'MARKET_EVENTS' in globals():
            for event_name, info in MARKET_EVENTS.items():
                ev_start = pd.to_datetime(info['start'])
                ev_end = pd.to_datetime(info['end'])
                if ev_start <= latest_date and ev_end >= earliest_date:
                    ax.axvspan(ev_start, ev_end, color=info['color'], alpha=0.18)
                    mid_date = max(ev_start, earliest_date) + (min(ev_end, latest_date) - max(ev_start, earliest_date)) / 2
                    ax.text(
                        mid_date, ax.get_ylim()[1] * 0.85, event_name, rotation=90, 
                        fontsize=8, fontweight='bold', ha='center', va='top',
                        bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="gray", alpha=0.75)
                    )

        ax.set_xlim(earliest_date, latest_date)
        ax.set_yscale('log')
        
        longest_clean = longest_lived_ticker.replace('.us.txt', '').upper()
        ax.set_title(
            f'CO-MOVEMENT BUCKET #{bucket_id} | Company Page {current_page} of {total_pages} '
            f'(Showing Companies {start_idx + 1}–{end_idx} of {total_companies_in_bucket})', 
            fontsize=14, fontweight='bold', pad=15
        )
        ax.set_xlabel('Timeline Year', fontsize=12)
        ax.set_ylabel('Re-indexed Price (Base 100, Log Scale)', fontsize=11)
        ax.grid(True, linestyle=':', alpha=0.6)
        ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1), fontsize=9, title=f"Companies (Page {current_page}/{total_pages})", framealpha=0.95)

        plt.tight_layout()
        plt.show()

# ==============================================================================
# CONTROLLABLE PAGINATION INPUTS
# ==============================================================================
START_BUCKET = 1        # Which bucket to view (e.g. 1, 2, 3...)
NUM_BUCKETS = 1         # How many buckets to render
COMPANY_PAGE = 1        # Page number for companies inside the bucket (Page 1, 2, 3...)
COMPANIES_PER_PAGE = 10 # Number of companies displayed per page

run_bucket_visualizer_paginated(
    start_bucket=START_BUCKET, 
    num_buckets=NUM_BUCKETS, 
    company_page=COMPANY_PAGE, 
    companies_per_page=COMPANIES_PER_PAGE
)


In [ ]:
# ==============================================================================
# LEAD-LAG CROSS-CORRELATION ANALYSIS (WHO MOVES FIRST?)
# ==============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def analyze_lead_lag(ticker_A, ticker_B, prices_df, max_lag_days=10):
    """
    Determines if ticker_A leads ticker_B or vice versa by calculating
    cross-correlation across time lags (-max_lag to +max_lag days).
    """
    if ticker_A not in prices_df.columns or ticker_B not in prices_df.columns:
        print("One or both tickers missing from dataset.")
        return
        
    # Extract log returns
    rA = np.log(prices_df[ticker_A] / prices_df[ticker_A].shift(1))
    rB = np.log(prices_df[ticker_B] / prices_df[ticker_B].shift(1))
    
    combined = pd.DataFrame({'A': rA, 'B': rB}).dropna()
    
    lags = range(-max_lag_days, max_lag_days + 1)
    corrs = []
    
    for lag in lags:
        if lag < 0:
            # A is shifted back (A leads B)
            c = combined['A'].shift(-lag).corr(combined['B'])
        elif lag > 0:
            # B is shifted back (B leads A)
            c = combined['A'].corr(combined['B'].shift(lag))
        else:
            c = combined['A'].corr(combined['B'])
        corrs.append(c)
        
    name_A = ticker_A.replace('.us.txt', '').upper()
    name_B = ticker_B.replace('.us.txt', '').upper()
    
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(lags, corrs, color=np.where(np.array(lags) == 0, '#1f77b4', '#ff7f0e'), alpha=0.8)
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    
    best_lag = lags[np.argmax(corrs)]
    max_c = max(corrs)
    
    ax.set_title(f'Lead-Lag Cross Correlation: {name_A} vs. {name_B}\nPeak Corr = {max_c:.2f} at Lag = {best_lag} Days', 
                 fontsize=12, fontweight='bold')
    ax.set_xlabel(f'← {name_A} Leads ({name_B} follows)  |  Same Day (0)  |  {name_B} Leads ({name_A} follows) →', fontsize=10)
    ax.set_ylabel('Correlation Coefficient', fontsize=10)
    ax.grid(True, linestyle=':', alpha=0.5)
    
    plt.tight_layout()
    plt.show()

# Run Lead-Lag Analysis on two sample co-moving stocks
analyze_lead_lag('ibm.us.txt', 'ge.us.txt', pivoted_prices, max_lag_days=10)


In [ ]:
# ==============================================================================
# NETWORK GRAPH TOPOLOGY (MINIMUM SPANNING TREE OF HIDDEN LINKAGES)
# ==============================================================================
import networkx as nx
import matplotlib.pyplot as plt

def plot_stock_network_graph(corr_matrix, max_stocks=40):
    """
    Constructs a Minimum Spanning Tree (MST) network graph to visualize the 
    backbone of hidden direct and indirect company linkages.
    """
    # Sample top N stocks for clean visualization
    sample_tickers = corr_matrix.index[:max_stocks]
    sub_corr = corr_matrix.loc[sample_tickers, sample_tickers].fillna(0)
    
    # Distance matrix: D = sqrt(2 * (1 - r))
    dist_matrix = np.sqrt(np.maximum(0, 2 * (1 - sub_corr.values)))
    
    # Build Graph
    G = nx.Graph()
    for i, t1 in enumerate(sample_tickers):
        clean_t1 = t1.replace('.us.txt', '').upper()
        G.add_node(clean_t1)
        for j, t2 in enumerate(sample_tickers):
            if i < j:
                clean_t2 = t2.replace('.us.txt', '').upper()
                w = dist_matrix[i, j]
                G.add_edge(clean_t1, clean_t2, weight=w)
                
    # Extract Minimum Spanning Tree (MST) - The topological backbone
    mst = nx.minimum_spanning_tree(G)
    
    plt.figure(figsize=(14, 10))
    pos = nx.spring_layout(mst, seed=42, k=0.35)
    
    # Draw nodes sized by connectivity (degree)
    degrees = dict(mst.degree())
    node_sizes = [v * 350 for v in degrees.values()]
    
    nx.draw_networkx_nodes(mst, pos, node_size=node_sizes, node_color='#1f77b4', alpha=0.85)
    nx.draw_networkx_edges(mst, pos, width=1.8, edge_color='#7f7f7f', alpha=0.7)
    nx.draw_networkx_labels(mst, pos, font_size=9, font_weight='bold', font_color='white')
    
    plt.title('Stock Market Linkage Network (Minimum Spanning Tree Topology)\nLarger Nodes = Central Systemic Hub Companies', 
              fontsize=14, fontweight='bold', pad=15)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

# Render Network Linkage Graph
plot_stock_network_graph(master_corr_matrix, max_stocks=400)


In [ ]:
# ==============================================================================
# ROLLING CORRELATION & STRUCTURAL BREAK ENGINE
# ==============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def plot_rolling_correlation(ticker_A, ticker_B, prices_df, window_days=756):
    """
    Computes and plots rolling correlation (default: 3-Year / 756 trading days)
    between two companies over time with market event overlays.
    """
    if ticker_A not in prices_df.columns or ticker_B not in prices_df.columns:
        print("One or both tickers missing from dataset.")
        return

    # Calculate log returns
    rA = np.log(prices_df[ticker_A] / prices_df[ticker_A].shift(1))
    rB = np.log(prices_df[ticker_B] / prices_df[ticker_B].shift(1))
    
    returns_pair = pd.DataFrame({ticker_A: rA, ticker_B: rB}).dropna()
    
    # Calculate 3-Year Rolling Correlation
    rolling_corr = returns_pair[ticker_A].rolling(window=window_days).corr(returns_pair[ticker_B])
    
    name_A = ticker_A.replace('.us.txt', '').upper()
    name_B = ticker_B.replace('.us.txt', '').upper()
    
    fig, ax = plt.subplots(figsize=(15, 6))
    ax.plot(rolling_corr.index, rolling_corr, color='#1f77b4', linewidth=2.0, label=f'3-Year Rolling Correlation ({name_A} vs {name_B})')
    ax.axhline(0, color='black', linestyle='--', linewidth=0.8)
    ax.axhline(rolling_corr.mean(), color='red', linestyle=':', linewidth=1.2, label=f'Overall Mean Corr ({rolling_corr.mean():.2f})')
    
    # Overlay Market Events, Bubbles & Wars
    if 'MARKET_EVENTS' in globals():
        for event_name, info in MARKET_EVENTS.items():
            ev_start = pd.to_datetime(info['start'])
            ev_end = pd.to_datetime(info['end'])
            if ev_start <= rolling_corr.index.max() and ev_end >= rolling_corr.index.min():
                ax.axvspan(ev_start, ev_end, color=info['color'], alpha=0.18)

    ax.set_ylim(-1.05, 1.05)
    ax.set_title(f'Structural Relationship Evolution: Rolling Correlation ({name_A} vs. {name_B})', fontsize=14, fontweight='bold', pad=15)
    ax.set_xlabel('Timeline Year', fontsize=12)
    ax.set_ylabel('3-Year Rolling Correlation Coefficient', fontsize=11)
    ax.grid(True, linestyle=':', alpha=0.6)
    ax.legend(loc='upper left', fontsize=10, framealpha=0.9)
    
    plt.tight_layout()
    plt.show()

# Run Rolling Correlation on two stocks (e.g. IBM vs GE)
plot_rolling_correlation('ibm.us.txt', 'ge.us.txt', pivoted_prices, window_days=756)


In [ ]:
# ==============================================================================
# CRISIS DOWNSIDE RESILIENCE & MAXIMUM DRAWDOWN ANALYZER
# ==============================================================================
import pandas as pd
import numpy as np

def analyze_crisis_resilience(prices_df, start_date='2007-10-01', end_date='2009-03-01', top_n=15):
    """
    Calculates Maximum Drawdown (MDD) and Total Return for all active stocks 
    during a specific crisis window (default: 2008 Global Financial Crisis).
    """
    print(f"=== Crisis Resilience Analysis ({start_date} to {end_date}) ===")
    
    # Slice prices during the crisis window
    crisis_prices = prices_df.loc[start_date:end_date].dropna(axis=1, thresh=50)
    
    results = []
    
    for ticker in crisis_prices.columns:
        series = crisis_prices[ticker].dropna()
        if len(series) < 20:
            continue
            
        # Cumulative Max (Peak)
        cum_max = series.cummax()
        # Drawdown = (Current - Peak) / Peak
        drawdowns = (series - cum_max) / cum_max
        max_drawdown = drawdowns.min() # Most negative value
        
        total_return = (series.iloc[-1] - series.iloc[0]) / series.iloc[0]
        
        clean_name = ticker.replace('.us.txt', '').upper()
        results.append({
            'Company': clean_name,
            'Total_Return (%)': round(total_return * 100, 2),
            'Max_Drawdown (%)': round(max_drawdown * 100, 2),
            'Initial_Price ($)': round(series.iloc[0], 2),
            'Min_Price ($)': round(series.min(), 2)
        })
        
    res_df = pd.DataFrame(results).set_index('Company')
    
    # 1. Top Safe Havens (Least Negative Max Drawdown)
    safe_havens = res_df.sort_values(by='Max_Drawdown (%)', ascending=False).head(top_n)
    
    # 2. Most Fragile Stocks (Most Negative Max Drawdown)
    fragile_stocks = res_df.sort_values(by='Max_Drawdown (%)', ascending=True).head(top_n)
    
    print("\n--- TOP SAFE-HAVEN STOCKS (Least Crash Sensitivity) ---")
    display(safe_havens)
    
    print("\n--- MOST FRAGILE STOCKS (Deepest Crash Vulnerability) ---")
    display(fragile_stocks)

# Run 2008 Financial Crisis Resilience Analysis
analyze_crisis_resilience(pivoted_prices, start_date='2007-10-01', end_date='2009-03-01', top_n=10)


In [ ]:
# ==============================================================================
# LATENT MACRO FACTOR EXTRACTION VIA PRINCIPAL COMPONENT ANALYSIS (PCA)
# ==============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

def extract_macro_factors(prices_df, n_components=5):
    """
    Performs PCA on daily returns to isolate top unobserved macro factors
    driving asset prices across the entire market.
    """
    print(f"1. Performing PCA to extract Top {n_components} Latent Macro Factors...")
    
    # Clean log returns
    log_ret = np.log(prices_df / prices_df.shift(1))
    
    # Filter stocks with high completeness
    valid_cols = log_ret.count()[log_ret.count() >= len(log_ret) * 0.7].index
    clean_returns = log_ret[valid_cols].fillna(0)
    
    # Fit PCA
    pca = PCA(n_components=n_components)
    pca.fit(clean_returns)
    
    exp_var = pca.explained_variance_ratio_
    cum_var = np.cumsum(exp_var)
    
    # Visualization 1: Variance Explained Bar Chart
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].bar(range(1, n_components + 1), exp_var * 100, color='#1f77b4', alpha=0.85)
    axes[0].set_title('Variance Explained per Macro Factor', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Principal Component (Factor #)', fontsize=10)
    axes[0].set_ylabel('Variance Explained (%)', fontsize=10)
    axes[0].grid(True, linestyle=':', alpha=0.5)
    
    for i, v in enumerate(exp_var):
        axes[0].text(i + 1, (v * 100) + 0.5, f"{v*100:.1f}%", ha='center', fontweight='bold', fontsize=9)
        
    axes[1].plot(range(1, n_components + 1), cum_var * 100, color='#2ca02c', marker='o', linewidth=2)
    axes[1].set_title('Cumulative Market Variance Explained', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Number of Components Included', fontsize=10)
    axes[1].set_ylabel('Cumulative Variance (%)', fontsize=10)
    axes[1].grid(True, linestyle=':', alpha=0.5)
    
    plt.tight_layout()
    plt.show()
    
    # 2. Extract Principal Component Trajectories
    factor_returns = pca.transform(clean_returns)
    factor_df = pd.DataFrame(factor_returns, index=clean_returns.index, columns=[f'Factor_{i+1}' for i in range(n_components)])
    
    # Cumulative Growth Curves of the Top 3 Hidden Macro Drivers
    cum_factors = factor_df[['Factor_1', 'Factor_2', 'Factor_3']].cumsum()
    
    fig, ax = plt.subplots(figsize=(15, 6))
    ax.plot(cum_factors.index, cum_factors['Factor_1'], label='Factor 1 (Broad Market Trend)', linewidth=2.0)
    ax.plot(cum_factors.index, cum_factors['Factor_2'], label='Factor 2 (Interest Rate / Sector Rotation)', linewidth=1.5)
    ax.plot(cum_factors.index, cum_factors['Factor_3'], label='Factor 3 (Commodity / Geopolitical Risk)', linewidth=1.5)
    
    ax.set_title('Hidden Macro Drivers Uncovered by PCA (Cumulative Factors)', fontsize=14, fontweight='bold', pad=15)
    ax.set_xlabel('Timeline Year', fontsize=12)
    ax.set_ylabel('Cumulative Factor Loadings', fontsize=11)
    ax.grid(True, linestyle=':', alpha=0.6)
    ax.legend(loc='upper left', fontsize=10, framealpha=0.9)
    
    plt.tight_layout()
    plt.show()

# Run Macro Factor Extraction
extract_macro_factors(pivoted_prices, n_components=5)
